In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import DeltaTable

df_old = spark.read.table("db_catalog.silver.products_silver")
df_old.display()

product_id,product_name,category,brand,price,discounted_price,price_category
P0001,Clearly Its,Beauty,Nike,1868.54,1307.98,High
P0002,Production Clear,Beauty,Apple,587.13,410.99,Low
P0003,Culture Coach,Home,Revlon,1599.24,1119.47,High
P0004,Movement Part,Sports,LG,651.71,456.2,Low
P0005,Fact Name,Clothing,Samsung,1861.78,1303.25,High
P0006,Usually Stop,Toys,Adidas,936.36,655.45,Medium
P0007,Reveal Current,Sports,Adidas,1954.02,1367.81,High
P0008,Force Language,Beauty,Puma,1251.26,875.88,Medium
P0009,Stage Leg,Clothing,Samsung,1247.15,873.01,Medium
P0010,Leader Then,Sports,Sony,975.53,682.87,Medium


In [0]:
init_load_flag = not spark.catalog.tableExists(
    "db_catalog.gold.dim_products"
)

print("Initial Load:", init_load_flag)


# ============================================================
# 3. GOLD DELTA PATH
# ============================================================

gold_path = "abfss://gold@dbproject.dfs.core.windows.net/dim_products"


# ============================================================
# 4. INITIAL LOAD
# ============================================================

if init_load_flag:

    # Generate surrogate key
    window_spec = Window.orderBy("product_id")

    df_dim_products = (
        df_old
        .withColumn(
            "dim_product_key",
            row_number().over(window_spec)
        )
        .withColumn(
            "start_date",
            current_timestamp()
        )
        .withColumn(
            "end_date",
            lit(None).cast("timestamp")
        )
        .withColumn(
            "current_flag",
            lit(True)
        )
    )

    # Select final columns
    df_dim_products = df_dim_products.select(
        "dim_product_key",
        "product_id",
        "product_name",
        "category",
        "brand",
        "price",
        "discounted_price",
        "price_category",
        "start_date",
        "end_date",
        "current_flag"
    )

    # Write initial dimension
    df_dim_products.write \
        .format("delta") \
        .mode("overwrite") \
        .save(gold_path)


    # Register Delta location as Unity Catalog table
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS db_catalog.gold.dim_products
        USING DELTA
        LOCATION '{gold_path}'
    """)

    print("Initial SCD Type 2 load completed")

else:

    # --------------------------------------------------------
    # 5.1 Open existing Delta table
    # --------------------------------------------------------

    target = DeltaTable.forPath(
        spark,
        gold_path
    )


    # --------------------------------------------------------
    # 5.2 Read existing dimension
    # --------------------------------------------------------

    df_target = spark.read.table(
        "db_catalog.gold.dim_products"
    )


    # --------------------------------------------------------
    # 5.3 Find maximum surrogate key
    # --------------------------------------------------------

    max_key = (
        df_target
        .select(
            coalesce(
                max("dim_product_key"),
                lit(0)
            ).alias("max_key")
        )
        .collect()[0]["max_key"]
    )

    print("Maximum existing key:", max_key)


    # New keys will start from max_key + 1
    start_key = max_key + 1

    print("New keys start from:", start_key)


    # --------------------------------------------------------
    # 5.4 Get only CURRENT records from dimension
    # --------------------------------------------------------

    df_current = (
        df_target
        .filter(
            col("current_flag") == True
        )
        .select(
            "product_id",
            col("product_name").alias("old_product_name"),
            col("category").alias("old_category"),
            col("brand").alias("old_brand"),
            col("price").alias("old_price"),
            col("discounted_price").alias("old_discounted_price"),
            col("price_category").alias("old_price_category")
        )
    )


    # --------------------------------------------------------
    # 5.5 Compare Silver data with current Gold data
    # --------------------------------------------------------

    df_changes = (
        df_old.alias("source")
        .join(
            df_current.alias("target"),
            col("source.product_id") == col("target.product_id"),
            "left"
        )
    )


    # --------------------------------------------------------
    # 5.6 Identify NEW products
    # --------------------------------------------------------

    df_changes = df_changes.withColumn(
        "is_new",
        col("target.product_id").isNull()
    )


    # --------------------------------------------------------
    # 5.7 Identify CHANGED products
    # --------------------------------------------------------

    df_changes = df_changes.withColumn(
        "is_changed",
        (
            (~col("is_new"))
            &
            (
                ~col("source.product_name").eqNullSafe(
                    col("target.old_product_name")
                )
                |
                ~col("source.category").eqNullSafe(
                    col("target.old_category")
                )
                |
                ~col("source.brand").eqNullSafe(
                    col("target.old_brand")
                )
                |
                ~col("source.price").eqNullSafe(
                    col("target.old_price")
                )
                |
                ~col("source.discounted_price").eqNullSafe(
                    col("target.old_discounted_price")
                )
                |
                ~col("source.price_category").eqNullSafe(
                    col("target.old_price_category")
                )
            )
        )
    )


    # --------------------------------------------------------
    # 5.8 See what has changed
    # --------------------------------------------------------

    df_changes.select(
        col("source.product_id").alias("product_id"),
        col("source.product_name").alias("product_name"),
        col("source.category").alias("category"),
        col("source.brand").alias("brand"),
        col("source.price").alias("price"),
        col("source.discounted_price").alias("discounted_price"),
        col("source.price_category").alias("price_category"),
        "is_new",
        "is_changed"
    ).display()


    # ========================================================
    # 6. CLOSE OLD RECORDS
    # ========================================================

    df_changed = (
        df_changes
        .filter(
            col("is_changed") == True
        )
        .select(
            col("source.product_id").alias("product_id")
        )
    )


    # Update the CURRENT record of changed products
    (
        target.alias("target")
        .merge(
            df_changed.alias("source"),
            """
            target.product_id = source.product_id
            AND target.current_flag = true
            """
        )
        .whenMatchedUpdate(
            set={
                "end_date": "current_timestamp()",
                "current_flag": "false"
            }
        )
        .execute()
    )


    # ========================================================
    # 7. PREPARE NEW + CHANGED RECORDS FOR INSERT
    # ========================================================

    df_to_insert = (
        df_changes
        .filter(
            (col("is_new") == True)
            |
            (col("is_changed") == True)
        )
        .select(
            col("source.product_id").alias("product_id"),
            col("source.product_name").alias("product_name"),
            col("source.category").alias("category"),
            col("source.brand").alias("brand"),
            col("source.price").alias("price"),
            col("source.discounted_price").alias("discounted_price"),
            col("source.price_category").alias("price_category")
        )
    )


    # ========================================================
    # 8. GENERATE NEW SURROGATE KEYS
    # ========================================================

    window_spec = Window.orderBy("product_id")

    df_to_insert = (
        df_to_insert
        .withColumn(
            "row_num",
            row_number().over(window_spec)
        )
        .withColumn(
            "dim_product_key",
            col("row_num") + lit(start_key) - lit(1)
        )
    )


    # ========================================================
    # 9. ADD SCD TYPE 2 COLUMNS
    # ========================================================

    df_to_insert = (
        df_to_insert
        .withColumn(
            "start_date",
            current_timestamp()
        )
        .withColumn(
            "end_date",
            lit(None).cast("timestamp")
        )
        .withColumn(
            "current_flag",
            lit(True)
        )
    )


    # ========================================================
    # 10. SELECT FINAL COLUMN ORDER
    # ========================================================

    df_to_insert = df_to_insert.select(
        "dim_product_key",
        "product_id",
        "product_name",
        "category",
        "brand",
        "price",
        "discounted_price",
        "price_category",
        "start_date",
        "end_date",
        "current_flag"
    )


    # --------------------------------------------------------
    # See records that will be inserted
    # --------------------------------------------------------

    df_to_insert.display()


    # ========================================================
    # 11. INSERT NEW + CHANGED RECORDS
    # ========================================================

    df_to_insert.write \
        .format("delta") \
        .mode("append") \
        .save(gold_path)


    print("Incremental SCD Type 2 load completed")


Initial Load: False
Maximum existing key: 500
New keys start from: 501


product_id,product_name,category,brand,price,discounted_price,price_category,is_new,is_changed
P0001,Clearly Its,Beauty,Nike,1868.54,1307.98,High,false,false
P0002,Production Clear,Beauty,Apple,587.13,410.99,Low,false,false
P0003,Culture Coach,Home,Revlon,1599.24,1119.47,High,false,false
P0004,Movement Part,Sports,LG,651.71,456.2,Low,false,false
P0005,Fact Name,Clothing,Samsung,1861.78,1303.25,High,false,false
P0006,Usually Stop,Toys,Adidas,936.36,655.45,Medium,false,false
P0007,Reveal Current,Sports,Adidas,1954.02,1367.81,High,false,false
P0008,Force Language,Beauty,Puma,1251.26,875.88,Medium,false,false
P0009,Stage Leg,Clothing,Samsung,1247.15,873.01,Medium,false,false
P0010,Leader Then,Sports,Sony,975.53,682.87,Medium,false,false


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_product_key,product_id,product_name,category,brand,price,discounted_price,price_category,start_date,end_date,current_flag


Incremental SCD Type 2 load completed


In [0]:
%sql
select * from db_catalog.gold.dim_products

dim_product_key,product_id,product_name,category,brand,price,discounted_price,price_category,start_date,end_date,current_flag
1,P0001,Clearly Its,Beauty,Nike,1868.54,1307.98,High,2026-09-14T10:14:35.157Z,null,true
2,P0002,Production Clear,Beauty,Apple,587.13,410.99,Low,2026-09-14T10:14:35.157Z,null,true
3,P0003,Culture Coach,Home,Revlon,1599.24,1119.47,High,2026-09-14T10:14:35.157Z,null,true
4,P0004,Movement Part,Sports,LG,651.71,456.2,Low,2026-09-14T10:14:35.157Z,null,true
5,P0005,Fact Name,Clothing,Samsung,1861.78,1303.25,High,2026-09-14T10:14:35.157Z,null,true
6,P0006,Usually Stop,Toys,Adidas,936.36,655.45,Medium,2026-09-14T10:14:35.157Z,null,true
7,P0007,Reveal Current,Sports,Adidas,1954.02,1367.81,High,2026-09-14T10:14:35.157Z,null,true
8,P0008,Force Language,Beauty,Puma,1251.26,875.88,Medium,2026-09-14T10:14:35.157Z,null,true
9,P0009,Stage Leg,Clothing,Samsung,1247.15,873.01,Medium,2026-09-14T10:14:35.157Z,null,true
10,P0010,Leader Then,Sports,Sony,975.53,682.87,Medium,2026-09-14T10:14:35.157Z,null,true
